# Checkworthy Detection Tests
Tests various checkworthy detection methods against all `article*.json` files in the directory.

In [1]:
import sys
import os
import glob
import json
import logging
from typing import List, Dict

# Setup Paths
project_root = os.path.abspath('../../..') 
if project_root not in sys.path:
    sys.path.append(project_root)

# Import Backend Models
try:
    from common.models.api.redis_models import (
        Article, NLPResult, NLPOptions, Claim, SentenceScore
    )
    from microservices.nlp.models.base import NLPComponent
    print("Successfully loaded backend data structures.")
except ImportError as e:
    print(f"Import Failed: {e}")

# Logging
logging.basicConfig(level=logging.INFO, format='%(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger("CheckworthyTest")

Successfully loaded backend data structures.


In [2]:
import logging
import spacy
import re
from typing import List

# Local imports
from microservices.nlp.models.base import NLPComponent
from common.models.api.redis_models import Article, NLPOptions, NLPResult, SentenceScore

logger = logging.getLogger(__name__)

class Preprocessor(NLPComponent):
    """
    "Universal Janitor" Preprocessor.
    
    Strategies applied:
    1. Regex Cleaning: Removes distinct artifacts (Dates, UI buttons, Footers) using strict patterns.
    2. Footer Cutoff: Stops processing the text entirely once footer keywords are detected.
    3. Linguistic Filtering: Uses Spacy's POS tagger to remove short lines (< 7 tokens) 
       that look like sentences but lack verbs (e.g., "Politics", "Frank Gardner").
    """
    def __init__(self):
        logger.info("Preprocessor: Loading Spacy 'en_core_web_sm' model...")
        try:
            # CRITICAL CHANGE: We REMOVED 'tagger' and 'attribute_ruler' from the disable list.
            # We need them enabled so Spacy can identify Verbs vs Nouns.
            self.nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
        except OSError:
            logger.error("Spacy model not found. Run: python -m spacy download en_core_web_sm")
            raise

    def _clean_and_repair_structure(self, raw_text: str) -> str:
        """
        Phase 1: Regex Cleaning (The Janitor)
        Removes obvious garbage (UI, Dates, Footers) before Spacy sees it.
        """
        if not raw_text: return ""

        lines = raw_text.split('\n')
        cleaned_lines = []
        
        # --- STOP PATTERNS (The Footer Cutoff) ---
        # If we see these, the article is likely over. Stop reading immediately.
        footer_cutoff_pattern = re.compile(r'(?i)^('
            r'more from (the )?bbc|related (content|stories|topics)|up next|most popular|'
            r'have you read\?|more on geographies|license and republishing|content index|'
            r'bbc\.com help|privacy policy|about us|follow .* on|sign up for'
        r')')
        
        # --- KILL LISTS (Regex Filters) ---
        # 1. Time & Meta: Matches "10 hrs ago", "Updated 1 min ago", "7 MIN READ"
        time_meta_pattern = re.compile(r'(?i)^('
            r'\d+\s+(hour|minute|day|second|hr|min)s?\s+ago|'
            r'updated\s+.*|'
            r'\d+\s+min\s+read|'
            r'(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\s+\d{1,2},?\s+\d{4}'
        r')')

        # 2. Credits & Sources: Matches "Getty Images", "Analysis by", "Image: ..."
        credits_pattern = re.compile(r'(?i)^('
            r'(image|photo|source|graphic|credits?):|'
            r'(left|right|top|bottom):|'
            r'analysis by|'
            r'this article is part of:|'
            r'unknown\.|'
            r'getty images|epa|afp|ugc|reuters|ap|copyright|davidoff studios'
        r')')

        # 3. UI Elements: Matches "Sign In", "Share", "Menu"
        ui_pattern = re.compile(r'(?i)^('
            r'register|sign in|log in|'
            r'skip to.*|'
            r'share|save|follow|subscribe|'
            r'menu|home|news|sport|weather|'
            r'listen to .* read this article|'
            r'loading\.\.\.|'
            r'create a free account|'
            r'terms of use'
        r')')

        # 4. Bylines: Matches "By [Name]" or "Correspondent"
        byline_pattern = re.compile(r'(?i)^('
            r'by\s+[A-Z][a-z]+\s+[A-Z][a-z]+|'
            r'.*correspondent.*|'
            r'writer,.*'
        r')')

        for line in lines:
            line = line.strip()
            
            # Basic Filtering
            if not line: continue 
            if len(line) < 4: continue # Catches very short noise like "EPA."
            
            # --- PHASE 2: CUTOFF CHECK ---
            if footer_cutoff_pattern.search(line):
                # Stop processing the rest of the file
                break

            # --- PHASE 3: REGEX FILTERING ---
            if ui_pattern.search(line): continue
            if time_meta_pattern.search(line): continue
            if credits_pattern.search(line): continue
            if byline_pattern.search(line) and len(line) < 50: continue

            # --- PHASE 4: STRUCTURAL REPAIR ---
            # If a line is a header/claim (no punctuation), force a period.
            # This ensures Spacy splits it from the next line.
            if line[-1] not in ".?!:;\"'":
                line += "."
                
            cleaned_lines.append(line)
            
        return " ".join(cleaned_lines)

    def run(self, article: Article, result: NLPResult, options: NLPOptions) -> None:
        # 1. Regex Cleaning
        raw_text = getattr(article, 'text', getattr(article, 'content', ""))
        clean_text = self._clean_and_repair_structure(raw_text)
        
        if not clean_text:
            logger.warning("Preprocessor: Text was empty after cleaning.")
            result.sentences = []
            return

        # 2. Spacy Analysis (Tokenization + POS Tagging)
        doc = self.nlp(clean_text)
        
        sentence_objects = []
        for idx, span in enumerate(doc.sents):
            text_segment = span.text.strip()
            
            # --- PHASE 5: LINGUISTIC FILTER ---
            # If a sentence is short (< 7 tokens), we strictly check its grammar.
            token_count = len(span)
            
            if token_count < 7:
                # Rule A: Keep Questions (e.g. "What next?") even if they lack verbs
                if "?" in text_segment:
                    pass 
                
                # Rule B: If it's not a question, it MUST have a Verb (VERB) or Auxiliary Verb (AUX)
                # This kills: "Pablo Uchoa." (PROPN), "Geographies in Depth." (NOUN)
                # This keeps: "He died." (VERB), "It was chaos." (AUX)
                else:
                    has_verb = any(token.pos_ in ["VERB", "AUX"] for token in span)
                    if not has_verb:
                        continue # Skip this sentence

            # Create sentence object
            s_obj = SentenceScore(
                index=idx, 
                text=text_segment, 
                score=0.0, 
                embedding=None
            )
            sentence_objects.append(s_obj)

        result.sentences = sentence_objects
        logger.info(f"Preprocessor: Cleaned & Split. Result: {len(sentence_objects)} sentences.")

print("--- Running Preprocessor Test ---")
try:
    pre = Preprocessor()
    pre.run(article, result, options)
    
    print(f"Success! Split into {len(result.sentences)} sentences.")
    for s in result.sentences:
        print(f"  [{s.index}] {s.text}")
        
except Exception as e:
    print(f"Error: {e}")

__main__ - INFO - Preprocessor: Loading Spacy 'en_core_web_sm' model...


--- Running Preprocessor Test ---
Error: name 'article' is not defined


## Load All Articles
Load all `article*.json` files from the test directory to create a comprehensive test pool.

In [3]:
# Find all article JSON files
json_files = sorted(glob.glob("article*.json"))
print(f"Found {len(json_files)} article files: {json_files}")

# Load all articles into a list
articles = []
article_metadata = []

for fpath in json_files:
    try:
        with open(fpath, 'r') as f:
            data = json.load(f)
        
        # Create Article object
        article = Article(
            title=data.get('article_title', 'Unknown Title'),
            text=data.get('article_text', ''),
            link=data.get('article_url', ''),
            summary=data.get('article_summary', '')
        )
        
        articles.append(article)
        article_metadata.append({
            'filename': fpath,
            'title': article.title,
            'text_length': len(article.text),
            'has_summary': bool(article.summary)
        })
        
        print(f"✓ Loaded: {fpath} - {article.title[:60]}...")
        
    except Exception as e:
        print(f"✗ Error loading {fpath}: {e}")

print(f"\n{'='*60}")
print(f"Successfully loaded {len(articles)} articles")
print(f"{'='*60}")

# Display summary statistics
if article_metadata:
    total_chars = sum(m['text_length'] for m in article_metadata)
    avg_chars = total_chars // len(article_metadata)
    print(f"\nTotal text: {total_chars:,} characters")
    print(f"Average article length: {avg_chars:,} characters")
    print(f"Articles with summaries: {sum(m['has_summary'] for m in article_metadata)}")

Found 6 article files: ['article.json', 'article1.json', 'article2.json', 'article3.json', 'article4.json', 'article5.json']
✓ Loaded: article.json - title...
✓ Loaded: article1.json - What could happen if the US strikes Iran? Here are seven sce...
✓ Loaded: article2.json - What next for Venezuela? What leaders and experts said at Da...
✓ Loaded: article3.json - FBI raids Georgia election office over 2020 voter fraud clai...
✓ Loaded: article4.json - AI 'slop' is transforming social media - and a backlash is b...
✓ Loaded: article5.json - New files deepen a critical mystery about those who partied ...

Successfully loaded 6 articles

Total text: 50,969 characters
Average article length: 8,494 characters
Articles with summaries: 6


## Display Article Details
Quick overview of each loaded article.

In [4]:
# Display detailed information for each article
for idx, (article, meta) in enumerate(zip(articles, article_metadata), 1):
    print(f"\n{'='*60}")
    print(f"Article {idx}: {meta['filename']}")
    print(f"{'='*60}")
    print(f"Title: {article.title}")
    print(f"Link: {article.link}")
    print(f"Text Length: {meta['text_length']:,} characters")
    
    # Show first 200 characters of text
    text_preview = article.text[:200].replace('\n', ' ')
    print(f"Text Preview: {text_preview}...")
    
    if article.summary:
        summary_preview = article.summary[:150].replace('\n', ' ')
        print(f"Summary: {summary_preview}...")


Article 1: article.json
Title: title
Link: url
Text Length: 4 characters
Text Preview: text...
Summary: summary...

Article 2: article1.json
Title: What could happen if the US strikes Iran? Here are seven scenarios
Link: https://www.bbc.com/news/articles/ce3kenge1k9o
Text Length: 9,170 characters
Text Preview: Skip to content Register Sign In What could happen if the US strikes Iran? Here are seven scenarios 6 days ago Share Save Frank Gardner Security correspondent EPA US President Donald Trump and Iran's ...
Summary: From regime change to retaliation, the BBC's Frank Gardner outlines possible outcomes of US strikes on Iran....

Article 3: article2.json
Title: What next for Venezuela? What leaders and experts said at Davos
Link: https://www.weforum.org/stories/2026/01/venezuela-what-next/
Text Length: 9,670 characters
Text Preview: GEOGRAPHIES IN DEPTH What next for Venezuela? What leaders and experts said at Davos Jan 23, 2026  Ngaire Woods: The international community has to 'creat

## Heuristic approach

In [5]:
import spacy
from microservices.nlp.models.base import NLPComponent
from common.models.api.redis_models import Article, NLPOptions, NLPResult

class HeuristicCheckWorthy(NLPComponent):
    """
    APPROACH 1: Rule-based tagging using Spacy.
    Flags sentences containing specific entities, numbers, and verbs.
    """
    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")
        self.reporting_verbs = {"say", "claim", "state", "report", "increase", "decrease", "cost", "kill"}
        
    def run(self, article: Article, result: NLPResult, options: NLPOptions) -> None:
        for s in result.sentences:
            doc = self.nlp(s.text)
            
            has_entity = any(ent.label_ in ["PERSON", "ORG", "GPE", "LOC"] for ent in doc.ents)
            has_number = any(ent.label_ in ["MONEY", "PERCENT", "CARDINAL", "DATE"] for ent in doc.ents)
            has_verb = any(token.lemma_ in self.reporting_verbs for token in doc)
            
            # Simple scoring logic based on presence of key elements
            score = 0.0
            if has_entity: score += 0.3
            if has_number: score += 0.4
            if has_verb: score += 0.3
                
            s.is_checkworthy = bool(score > 0.6)
            s.confidence = score

## Transformer

In [6]:
from transformers import pipeline
from microservices.nlp.models.base import NLPComponent
from common.models.api.redis_models import Article, NLPOptions, NLPResult

class TransformerCheckWorthy(NLPComponent):
    """
    APPROACH 2: Semantic classification using a Transformer model.
    Uses NLI (Natural Language Inference) to determine if a sentence constitutes a verifiable claim.
    """
    def __init__(self):
        # We use a lightweight zero-shot classifier as a proxy for a CheckThat-trained model
        self.classifier = pipeline(
            "zero-shot-classification", 
            model="cross-encoder/nli-distilroberta-base"
        )
        self.candidate_labels = ["a verifiable factual claim", "a subjective opinion or question"]
        self.threshold = 0.65

    def run(self, article: Article, result: NLPResult, options: NLPOptions) -> None:
        if not result.sentences:
            return
            
        texts = [s.text for s in result.sentences]
        
        # Batch process the sentences
        classifications = self.classifier(texts, self.candidate_labels)
        
        for i, s in enumerate(result.sentences):
            res = classifications[i]
            # Get the confidence score for the "verifiable claim" label
            claim_index = res['labels'].index("a verifiable factual claim")
            score = res['scores'][claim_index]
            
            s.is_checkworthy = bool(score >= self.threshold)
            s.confidence = score

/home/farhan/miniconda2/envs/nlp311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Prompt engineered

In [7]:
import json
from llama_cpp import Llama
from microservices.nlp.models.base import NLPComponent
from common.models.api.redis_models import Article, NLPOptions, NLPResult

class LLMCheckWorthy(NLPComponent):
    """
    APPROACH 3: Fast LLM Evaluation using GGUF quantized models (llama.cpp).
    Uses optimized binary format for 5-10x faster loading and 2-3x faster inference.
    """
    def __init__(self, model_path: str = "Qwen/Qwen2.5-1.5B-Instruct-GGUF", filename: str = "*q4_k_m.gguf"):
        """
        Initialize with a quantized GGUF model.
        
        Options:
        - "Qwen/Qwen2.5-1.5B-Instruct-GGUF" with "*q4_k_m.gguf" (lightweight, fast)
        - "Qwen/Qwen2.5-3B-Instruct-GGUF" with "*q4_k_m.gguf" (more capable)
        - "Qwen/Qwen2.5-7B-Instruct-GGUF" with "*q4_k_m.gguf" (best quality)
        
        Or provide local .gguf file path
        """
        print(f"Loading GGUF model from {model_path}...")
        
        # Check if it's a local file or HuggingFace repo
        if model_path.endswith('.gguf'):
            # Local file
            self.llm = Llama(
                model_path=model_path,
                n_ctx=512,  # Reduced context for lower memory usage
                n_threads=2,  # Reduced threads to prevent crash
                n_batch=128,  # Smaller batch size
                verbose=False
            )
        else:
            # Download from HuggingFace
            self.llm = Llama.from_pretrained(
                repo_id=model_path,
                filename=filename,
                n_ctx=512,  # Reduced context for lower memory usage
                n_threads=2,  # Reduced threads to prevent crash
                n_batch=128,  # Smaller batch size
                verbose=False
            )
        
        print("✓ GGUF model loaded and ready")

    def run(self, article: Article, result: NLPResult, options: NLPOptions) -> None:
        # To save tokens, we only send sentences longer than 5 words
        candidates = [s for s in result.sentences if len(s.text.split()) > 5]
        
        if not candidates:
            return
        
        # Smaller batch size due to reduced context window
        batch_size = 5
        
        for i in range(0, len(candidates), batch_size):
            batch = candidates[i:i+batch_size]
            sentences_json = json.dumps([s.text for s in batch])
            
            prompt = f"""You are a fact-checker. For each sentence, determine if it is "check-worthy" (factual claim about politics, health, science, or public interest).

Return ONLY a JSON object with a "scores" array of numbers 0.0-1.0.

Sentences: {sentences_json}

JSON:"""

            try:
                # Generate response using llama.cpp
                response = self.llm.create_chat_completion(
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.3,
                    max_tokens=200,
                    response_format={"type": "json_object"}
                )
                
                # Extract response text
                response_text = response['choices'][0]['message']['content'].strip()
                
                # Extract JSON from response (handles markdown code blocks)
                if "```json" in response_text:
                    response_text = response_text.split("```json")[1].split("```")[0].strip()
                elif "```" in response_text:
                    response_text = response_text.split("```")[1].split("```")[0].strip()
                
                # Parse JSON
                result_json = json.loads(response_text)
                scores = result_json.get("scores", [])
                
                # Assign scores to sentences
                for s, score in zip(batch, scores):
                    s.is_checkworthy = bool(score > 0.7)
                    s.confidence = float(score)
                    
            except (json.JSONDecodeError, KeyError, IndexError) as e:
                print(f"LLM Parsing Error: {e}")
                if 'response_text' in locals():
                    print(f"Response: {response_text[:200]}...")
                # Fallback: mark all as uncertain
                for s in batch:
                    s.is_checkworthy = False
                    s.confidence = 0.5
            except Exception as e:
                print(f"LLM Error: {e}")
                for s in batch:
                    s.is_checkworthy = False
                    s.confidence = 0.5

In [11]:
import os
import copy

# ==========================================
# 1. INITIALIZE COMPONENTS
# ==========================================
print("Initializing models... (This might take a minute for the Transformer and LLM)")

preprocessor = Preprocessor()
heuristic_model = HeuristicCheckWorthy()
transformer_model = TransformerCheckWorthy()

# Initialize LLM with GGUF quantized model (no API key needed)
try:
    llm_model = LLMCheckWorthy(model_path="Qwen/Qwen2.5-1.5B-Instruct-GGUF", filename="*q4_k_m.gguf")
except Exception as e:
    llm_model = None
    print(f"Notice: Could not load LLM model. Error: {e}")
    print("Skipping LLM evaluation.")

# ==========================================
# 2. EVALUATION LOOP
# ==========================================
# Test on a subset of articles to keep output readable (e.g., Article 1 and 2)
test_articles = articles
options = NLPOptions()

# Store results for CSV export
comparison_results = []

for idx, article in enumerate(test_articles):
    print(f"\n{'='*100}")
    print(f"EVALUATING ARTICLE {idx+1}: {article.title}")
    print(f"{'='*100}")
    
    # Setup base NLP Result and parse sentences
    base_result = NLPResult(sentences=[], entities_in_article=[], claims_in_article=[])
    preprocessor.run(article, base_result, options)
    
    if not base_result.sentences:
        print("No sentences found after preprocessing. Skipping.")
        continue
        
    # Take a sample of 15 sentences to keep the matrix clean
    sample_sentences = base_result.sentences[:15]
    
    # Deep copy the results so each model gets a clean slate to score
    res_heuristic = copy.deepcopy(base_result)
    res_heuristic.sentences = copy.deepcopy(sample_sentences)
    
    res_transformer = copy.deepcopy(base_result)
    res_transformer.sentences = copy.deepcopy(sample_sentences)
    
    res_llm = copy.deepcopy(base_result)
    res_llm.sentences = copy.deepcopy(sample_sentences)
    
    # --- RUN INFERENCE ---
    print("Running Heuristic Model...")
    heuristic_model.run(article, res_heuristic, options)
    
    print("Running Transformer Model...")
    transformer_model.run(article, res_transformer, options)
    
    if llm_model:
        print("Running LLM Model...")
        llm_model.run(article, res_llm, options)
    
    # --- PRINT COMPARATIVE MATRIX ---
    print(f"\n{'-'*110}")
    # Header
    print(f"{'HEURISTIC':<12} | {'TRANSFORMER':<12} | {'LLM':<12} | {'SENTENCE'}")
    print(f"{'-'*110}")
    
    for i in range(len(sample_sentences)):
        # Format: Score (T/F)
        h_flag = "T" if res_heuristic.sentences[i].is_checkworthy else "F"
        h_str = f"{res_heuristic.sentences[i].confidence:.2f} ({h_flag})"
        
        t_flag = "T" if res_transformer.sentences[i].is_checkworthy else "F"
        t_str = f"{res_transformer.sentences[i].confidence:.2f} ({t_flag})"
        
        if llm_model:
            l_flag = "T" if res_llm.sentences[i].is_checkworthy else "F"
            l_str = f"{res_llm.sentences[i].confidence:.2f} ({l_flag})"
        else:
            l_str = "N/A"
            l_flag = "N/A"
            
        # Truncate text so it fits neatly in the terminal
        text = sample_sentences[i].text.replace('\n', ' ')
        if len(text) > 65: 
            text = text[:62] + "..."
            
        print(f"{h_str:<12} | {t_str:<12} | {l_str:<12} | {text}")
        
        # Store result for CSV export
        comparison_results.append({
            'article_index': idx + 1,
            'article_title': article.title,
            'sentence_index': i,
            'sentence_text': sample_sentences[i].text,
            'heuristic_score': res_heuristic.sentences[i].confidence,
            'heuristic_checkworthy': res_heuristic.sentences[i].is_checkworthy,
            'transformer_score': res_transformer.sentences[i].confidence,
            'transformer_checkworthy': res_transformer.sentences[i].is_checkworthy,
            'llm_score': res_llm.sentences[i].confidence if llm_model else None,
            'llm_checkworthy': res_llm.sentences[i].is_checkworthy if llm_model else None
        })

print(f"\n\n{'='*100}")
print(f"Evaluation Complete! Collected {len(comparison_results)} sentence comparisons.")
print(f"{'='*100}")

__main__ - INFO - Preprocessor: Loading Spacy 'en_core_web_sm' model...


Initializing models... (This might take a minute for the Transformer and LLM)


httpx - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/nli-distilroberta-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cross-encoder/nli-distilroberta-base/b14d131f9d32668a5e6a982729b57ff6ed5dfcbd/config.json "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 215.56it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: cross-encoder/nli-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/cross-encoder/nli-distilroberta-base/tree/main/additional_chat_templates?recursive

Loading GGUF model from Qwen/Qwen2.5-1.5B-Instruct-GGUF...


/home/farhan/miniconda2/envs/nlp311/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
httpx - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF/resolve/main/qwen2.5-1.5b-instruct-q4_k_m.gguf "HTTP/1.1 302 Found"
huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
llama_context: n_ctx_per_seq (512) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
__main__ - INFO - Preprocessor: Cleaned & Split. Result: 0 sentences.


...


--------------------------------------------------------------------------------------------------------------
HEURISTIC    | TRANSFORMER  | LLM          | SENTENCE
--------------------------------------------------------------------------------------------------------------
0.00 (F)     | 0.20 (F)     | 0.00 (F)     | GEOGRAPHIES IN DEPTH.
0.30 (F)     | 0.13 (F)     | 0.00 (F)     | What next for Venezuela?
0.60 (F)     | 0.43 (F)     | 0.00 (F)     | What leaders and experts said at Davos.
0.00 (F)     | 0.58 (F)     | 0.00 (F)     | Ngaire Woods: The international community has to 'create the c...
0.00 (F)     | 0.53 (F)     | 0.00 (F)     | Image: World Economic Forum.
0.70 (T)     | 0.10 (F)     | 0.00 (F)     | Venezuela faces profound political and economic uncertainty in...
0.30 (F)     | 0.11 (F)     | 0.00 (F)     | Core questions remain unanswered, including the feasibility of...
0.70 (T)     | 0.15 (F)     | 0.00 (F)     | Venezuela’s political future and economic recover

__main__ - INFO - Preprocessor: Cleaned & Split. Result: 124 sentences.


Running Heuristic Model...
Running Transformer Model...
Running LLM Model...


__main__ - INFO - Preprocessor: Cleaned & Split. Result: 79 sentences.



--------------------------------------------------------------------------------------------------------------
HEURISTIC    | TRANSFORMER  | LLM          | SENTENCE
--------------------------------------------------------------------------------------------------------------
0.30 (F)     | 0.14 (F)     | 0.00 (F)     | AI 'slop' is transforming social media - and a backlash is bre...
0.30 (F)     | 0.31 (F)     | 0.00 (F)     | Théodore remembers the AI slop that tipped him over the edge.
0.40 (F)     | 0.17 (F)     | 0.00 (F)     | The image was of two emaciated, impoverished South Asian child...
0.00 (F)     | 0.17 (F)     | 0.00 (F)     | For some reason, despite their boyish features they have thick...
0.40 (F)     | 0.20 (F)     | 0.00 (F)     | One of them had no hands and only one foot.
0.30 (F)     | 0.22 (F)     | 0.00 (F)     | The other was holding a sign saying it's his birthday and aski...
0.00 (F)     | 0.16 (F)     | 0.00 (F)     | Inexplicably they are sitting in the m

## Export Results to CSV
Export the comparison results to a CSV file for further analysis.

In [12]:
import pandas as pd
from datetime import datetime

# Convert results to DataFrame
df = pd.DataFrame(comparison_results)

# Generate filename with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_filename = f"checkworthy_comparison_{timestamp}.csv"

# Save to CSV
df.to_csv(csv_filename, index=False)

print(f"✓ Results exported to: {csv_filename}")
print(f"✓ Total rows: {len(df)}")
print(f"\nPreview:")
print(df.head(10))

✓ Results exported to: checkworthy_comparison_20260225_134126.csv
✓ Total rows: 75

Preview:
   article_index                                      article_title  \
0              2  What could happen if the US strikes Iran? Here...   
1              2  What could happen if the US strikes Iran? Here...   
2              2  What could happen if the US strikes Iran? Here...   
3              2  What could happen if the US strikes Iran? Here...   
4              2  What could happen if the US strikes Iran? Here...   
5              2  What could happen if the US strikes Iran? Here...   
6              2  What could happen if the US strikes Iran? Here...   
7              2  What could happen if the US strikes Iran? Here...   
8              2  What could happen if the US strikes Iran? Here...   
9              2  What could happen if the US strikes Iran? Here...   

   sentence_index                                      sentence_text  \
0               0          What could happen if the U

## Compare Heuristic Models
Compare the original `HeuristicCheckWorthy` vs the refined `HeuristicCheckWorthy2` (with speculation filtering) to see the impact of the penalty for speculative language.

In [13]:
import spacy
from typing import Set
from microservices.nlp.models.base import NLPComponent
from common.models.api.redis_models import Article, NLPOptions, NLPResult

class HeuristicCheckWorthy2(NLPComponent):
    """
    APPROACH 1 (REFINED): Rule-based tagging with Speculation Filtering.
    
    Scoring Logic:
    +0.3: Has Entity (Person, Org, GPE)
    +0.4: Has Number/Date (The core of most facts)
    +0.3: Has Reporting Verb (said, claimed)
    -0.4: Has Speculative Verb (could, might, poised) -> THE PENALTY
    """
    def __init__(self):
        # Disable heavy components we don't need for speed
        self.nlp = spacy.load("en_core_web_sm", disable=["textcat"])
        
        # Verbs that indicate a claim is being made
        self.reporting_verbs = {
            "say", "claim", "state", "report", "announce", "confirm", 
            "reveal", "admit", "warn", "accuse"
        }
        
        # Verbs that indicate the sentence is a prediction or guess (Not a fact)
        self.speculative_verbs = {
            "could", "would", "might", "may", "will", "can", "should",
            "predict", "expect", "poised", "likely", "possible", "potential",
            "future", "scenario", "imagine", "if", "appear", "seem"
        }

    def run(self, article: Article, result: NLPResult, options: NLPOptions) -> None:
        if not result.sentences:
            return

        for s in result.sentences:
            doc = self.nlp(s.text)
            
            # 1. POSITIVE SIGNALS
            has_entity = any(ent.label_ in ["PERSON", "ORG", "GPE", "LOC", "EVENT"] for ent in doc.ents)
            has_number = any(ent.label_ in ["MONEY", "PERCENT", "CARDINAL", "DATE", "TIME"] for ent in doc.ents)
            
            # Check lemmas for reporting verbs
            has_reporting = any(token.lemma_ in self.reporting_verbs for token in doc)
            
            # 2. NEGATIVE SIGNALS (The Filter)
            # Check for speculative keywords
            is_speculative = any(token.lemma_ in self.speculative_verbs for token in doc)
            
            # 3. SCORING
            score = 0.0
            
            if has_number: score += 0.4  # Numbers are high value
            if has_entity: score += 0.3
            if has_reporting: score += 0.2
            
            # Apply Penalty
            if is_speculative:
                score -= 0.4
                
            # Clamp score between 0 and 1
            score = max(0.0, min(1.0, score))
            
            # 4. DECISION
            # Threshold 0.5 means it needs at least (Number + Entity) or (Number + Verb - Speculation)
            s.is_checkworthy = bool(score >= 0.5)
            s.confidence = score

In [14]:
import copy

# ==========================================
# HEURISTIC COMPARISON SETUP
# ==========================================
print("Comparing Heuristic Models...")
print("=" * 100)

# Initialize both heuristic models
heuristic_v1 = HeuristicCheckWorthy()
heuristic_v2 = HeuristicCheckWorthy2()

# Test on all articles
test_articles_comparison = articles
options = NLPOptions()

# Store results for comparison
heuristic_comparison_results = []

for idx, article in enumerate(test_articles_comparison):
    print(f"\n{'='*100}")
    print(f"ARTICLE {idx+1}: {article.title}")
    print(f"{'='*100}")
    
    # Setup base NLP Result and parse sentences
    base_result = NLPResult(sentences=[], entities_in_article=[], claims_in_article=[])
    preprocessor.run(article, base_result, options)
    
    if not base_result.sentences:
        print("No sentences found after preprocessing. Skipping.")
        continue
        
    # Take a sample of 15 sentences to keep output readable
    sample_sentences = base_result.sentences[:15]
    
    # Deep copy for each model
    res_v1 = copy.deepcopy(base_result)
    res_v1.sentences = copy.deepcopy(sample_sentences)
    
    res_v2 = copy.deepcopy(base_result)
    res_v2.sentences = copy.deepcopy(sample_sentences)
    
    # Run both heuristic models
    heuristic_v1.run(article, res_v1, options)
    heuristic_v2.run(article, res_v2, options)
    
    # Print comparison matrix
    print(f"\n{'-'*120}")
    print(f"{'HEURISTIC V1':<15} | {'HEURISTIC V2':<15} | {'DIFF':<10} | {'SENTENCE'}")
    print(f"{'-'*120}")
    
    differences_count = 0
    
    for i in range(len(sample_sentences)):
        # Format scores
        v1_flag = "T" if res_v1.sentences[i].is_checkworthy else "F"
        v1_str = f"{res_v1.sentences[i].confidence:.2f} ({v1_flag})"
        
        v2_flag = "T" if res_v2.sentences[i].is_checkworthy else "F"
        v2_str = f"{res_v2.sentences[i].confidence:.2f} ({v2_flag})"
        
        # Calculate difference
        score_diff = res_v2.sentences[i].confidence - res_v1.sentences[i].confidence
        
        # Mark if classification changed
        changed = "✓" if v1_flag != v2_flag else ""
        diff_str = f"{score_diff:+.2f} {changed}"
        
        if v1_flag != v2_flag:
            differences_count += 1
        
        # Truncate text
        text = sample_sentences[i].text.replace('\n', ' ')
        if len(text) > 55:
            text = text[:52] + "..."
        
        print(f"{v1_str:<15} | {v2_str:<15} | {diff_str:<10} | {text}")
        
        # Store for CSV
        heuristic_comparison_results.append({
            'article_index': idx + 1,
            'article_title': article.title,
            'sentence_index': i,
            'sentence_text': sample_sentences[i].text,
            'heuristic_v1_score': res_v1.sentences[i].confidence,
            'heuristic_v1_checkworthy': res_v1.sentences[i].is_checkworthy,
            'heuristic_v2_score': res_v2.sentences[i].confidence,
            'heuristic_v2_checkworthy': res_v2.sentences[i].is_checkworthy,
            'score_difference': score_diff,
            'classification_changed': v1_flag != v2_flag
        })
    
    print(f"\nClassification changes: {differences_count}/{len(sample_sentences)}")

print(f"\n\n{'='*100}")
print(f"Heuristic Comparison Complete!")
print(f"Total sentences compared: {len(heuristic_comparison_results)}")
print(f"{'='*100}")

Comparing Heuristic Models...


__main__ - INFO - Preprocessor: Cleaned & Split. Result: 0 sentences.
__main__ - INFO - Preprocessor: Cleaned & Split. Result: 63 sentences.



ARTICLE 1: title
No sentences found after preprocessing. Skipping.

ARTICLE 2: What could happen if the US strikes Iran? Here are seven scenarios


__main__ - INFO - Preprocessor: Cleaned & Split. Result: 51 sentences.



------------------------------------------------------------------------------------------------------------------------
HEURISTIC V1    | HEURISTIC V2    | DIFF       | SENTENCE
------------------------------------------------------------------------------------------------------------------------
0.30 (F)        | 0.00 (F)        | -0.30      | What could happen if the US strikes Iran?
0.40 (F)        | 0.00 (F)        | -0.40      | Here are seven scenarios.
0.30 (F)        | 0.30 (F)        | +0.00      | US President Donald Trump and Iran's Supreme Leader ...
0.70 (T)        | 0.30 (F)        | -0.40 ✓    | The US appears poised to strike Iran within days.
0.00 (F)        | 0.00 (F)        | +0.00      | While the potential targets are largely predictable,...
0.30 (F)        | 0.00 (F)        | -0.30      | So, if no last-minute deal can be reached with Tehra...
0.40 (F)        | 0.40 (F)        | +0.00      | 1. Targeted, surgical strikes, minimal civilian casu...
0.30 (F)      

__main__ - INFO - Preprocessor: Cleaned & Split. Result: 26 sentences.



------------------------------------------------------------------------------------------------------------------------
HEURISTIC V1    | HEURISTIC V2    | DIFF       | SENTENCE
------------------------------------------------------------------------------------------------------------------------
0.00 (F)        | 0.00 (F)        | +0.00      | GEOGRAPHIES IN DEPTH.
0.30 (F)        | 0.30 (F)        | +0.00      | What next for Venezuela?
0.60 (F)        | 0.50 (T)        | -0.10 ✓    | What leaders and experts said at Davos.
0.00 (F)        | 0.00 (F)        | +0.00      | Ngaire Woods: The international community has to 'cr...
0.00 (F)        | 0.00 (F)        | +0.00      | Image: World Economic Forum.
0.70 (T)        | 0.70 (T)        | +0.00      | Venezuela faces profound political and economic unce...
0.30 (F)        | 0.30 (F)        | +0.00      | Core questions remain unanswered, including the feas...
0.70 (T)        | 0.30 (F)        | -0.40 ✓    | Venezuela’s political f

__main__ - INFO - Preprocessor: Cleaned & Split. Result: 124 sentences.
__main__ - INFO - Preprocessor: Cleaned & Split. Result: 79 sentences.



------------------------------------------------------------------------------------------------------------------------
HEURISTIC V1    | HEURISTIC V2    | DIFF       | SENTENCE
------------------------------------------------------------------------------------------------------------------------
0.30 (F)        | 0.30 (F)        | +0.00      | AI 'slop' is transforming social media - and a backl...
0.30 (F)        | 0.30 (F)        | +0.00      | Théodore remembers the AI slop that tipped him over ...
0.40 (F)        | 0.40 (F)        | +0.00      | The image was of two emaciated, impoverished South A...
0.00 (F)        | 0.00 (F)        | +0.00      | For some reason, despite their boyish features they ...
0.40 (F)        | 0.40 (F)        | +0.00      | One of them had no hands and only one foot.
0.30 (F)        | 0.20 (F)        | -0.10      | The other was holding a sign saying it's his birthda...
0.00 (F)        | 0.00 (F)        | +0.00      | Inexplicably they are sitting in

### Export Heuristic Comparison to CSV

In [16]:
import pandas as pd
from datetime import datetime

# Convert heuristic comparison results to DataFrame
df_heuristic = pd.DataFrame(heuristic_comparison_results)

# Generate filename with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_filename_heuristic = f"heuristic_comparison_{timestamp}.csv"

# Save to CSV
df_heuristic.to_csv(csv_filename_heuristic, index=False)

print(f"✓ Heuristic comparison exported to: {csv_filename_heuristic}")
print(f"✓ Total rows: {len(df_heuristic)}")

# Show summary statistics
total_changes = df_heuristic['classification_changed'].sum()
total_sentences = len(df_heuristic)
change_percentage = (total_changes / total_sentences) * 100

print(f"\n{'='*60}")
print(f"Summary Statistics:")
print(f"{'='*60}")
print(f"Total sentences analyzed: {total_sentences}")
print(f"Classification changes: {total_changes} ({change_percentage:.1f}%)")
print(f"Average V1 score: {df_heuristic['heuristic_v1_score'].mean():.3f}")
print(f"Average V2 score: {df_heuristic['heuristic_v2_score'].mean():.3f}")
print(f"Average score difference: {df_heuristic['score_difference'].mean():.3f}")

print(f"\n{'='*60}")
print(f"Preview of sentences where classification changed:")
print(f"{'='*60}")
changed_sentences = df_heuristic[df_heuristic['classification_changed'] == True]
if len(changed_sentences) > 0:
    print(changed_sentences[['sentence_text', 'heuristic_v1_score', 'heuristic_v2_score', 'score_difference']].head(10))
else:
    print("No classification changes detected.")

✓ Heuristic comparison exported to: heuristic_comparison_20260225_134717.csv
✓ Total rows: 75

Summary Statistics:
Total sentences analyzed: 75
Classification changes: 8 (10.7%)
Average V1 score: 0.407
Average V2 score: 0.352
Average score difference: -0.055

Preview of sentences where classification changed:
                                        sentence_text  heuristic_v1_score  \
3   The US appears poised to strike Iran within days.                 0.7   
17            What leaders and experts said at Davos.                 0.6   
22  Venezuela’s political future and economic reco...                 0.7   
24  And in Davos this week, the country's future w...                 0.7   
28  No election has been announced, despite consti...                 0.4   
32  In a statement to Reuters, the FBI said it was...                 0.6   
37  Agents with FBI vests were seen entering and e...                 0.7   
39  "This is an assault on your vote," Fulton Coun...                 0.6